In [1]:
# Cell 1: Chọn device phù hợp (NVIDIA / AMD / CPU)

import math
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from matrix_correla import (
    load_traffic_tensor,
    compute_anchor_correlations,
    build_influenced_subgraph,
    build_laplacian_from_distance,
    build_zone
)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA-compatible GPU (NVIDIA hoặc AMD qua ROCm):", torch.cuda.get_device_name(0))
else:
    # Thử DirectML (Windows + AMD)
    try:
        import torch_directml
        device = torch_directml.device()
        print("Using DirectML device (có thể là AMD Radeon qua DirectML).")
    except ImportError:
        device = torch.device("cpu")
        print("Không tìm thấy GPU, dùng CPU.")

device


Using DirectML device (có thể là AMD Radeon qua DirectML).


device(type='privateuseone', index=0)

In [2]:
class TemporalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.d_model = d_model
        self.rel_pos_emb = nn.Embedding(max_len, d_model)
        self.time_proj = nn.Linear(4,d_model)
    def forward(self,time_of_day,day_of_week):
        B,L = time_of_day.shape

        # --- relative position embedding ---
        rel_pos = torch.arange(L, device=time_of_day.device) #(L,)
        rel_pos = rel_pos.unsqueeze(0).expand(B,L) # biến thành (1,L) rồi nhân bản thành (B,L)
        rel_pos_pe = self.rel_pos_emb(rel_pos)

        # --- sin/cos time-of-day (0..1) -> (0..2π) ---
        tod_angle = time_of_day * 2 *math.pi
        tod_sin = torch.sin(tod_angle)
        tod_cos = torch.cos(tod_angle)

        # --- sin/cos day-of-week (0..6) ---
        dow = day_of_week.float() / 5.0
        dow_angle = dow * 2 * math.pi
        dow_sin = torch.sin(dow_angle)
        dow_cos = torch.cos(dow_angle)
        t_feat = torch.stack([tod_sin, tod_cos, dow_sin, dow_cos], dim=-1)
        # project lên d_model
        t_pe = self.time_proj(t_feat)
        return rel_pos_pe + t_pe

In [3]:
class SpatialEncoding(nn.Module):
    """
    Spatial PE = Laplacian eigenvectors của subgraph.
    eigvec: (N, d_spa)
    """
    def __init__(self,d_model,d_spa):
        super().__init__()
        self.proj = nn.Linear(d_spa, d_model)
    def forward(self, lap_eigvec):
        """
        lap_eigvec: (N, d_spa)
        return: (N, d_model)
        """
        return self.proj(lap_eigvec)  # (N, d_model)


In [4]:
class SpatioTemporalEncoder(nn.Module):
    """
    Encoder-only Transformer với token = (segment, time).

    Input x: (B, L, N, d_in)
    Output H: (B, L, N, d_model)
    """
    def __init__(self, d_in, d_model, nhead,num_layers,dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(d_in, d_model)
        self.temp_enc = TemporalEncoding(d_model=d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,dim_feedforward=dim_feedforward,dropout=dropout,batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer,num_layers=num_layers)

    def forward(self, x, time_of_day, day_of_week, spatial_pe, node_mask=None):
        B,L,N,d_in = x.shape
        x_proj = self.input_proj(x)

        t_pe = self.temp_enc(time_of_day, day_of_week).unsqueeze(2)
        s_pe = spatial_pe.unsqueeze(1)
        h = x_proj + t_pe + s_pe                      # (B,L,N,D)

        h = h.reshape(B, L*N, self.d_model)           # (B,S,D)

        src_kpm = None
        if node_mask is not None:
            pad_nodes  = (node_mask < 0.5)            # (B,N)
            src_kpm = pad_nodes.unsqueeze(1).expand(B,L,N).reshape(B, L*N)

        H = self.encoder(h, src_key_padding_mask=src_kpm)  # batch_first=True
        return H.reshape(B, L, N, self.d_model)




In [5]:
# Cell 5: Prediction Head + Full SpatioTemporalTransformer

class PredictionHead(nn.Module):
    def __init__(self, d_model, hidden_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, H_last):      # H_last: (B, N, d_model)
        B, N, D = H_last.shape
        logits = self.mlp(H_last.reshape(B*N, D))  # (B*N,1)
        return logits.reshape(B, N)                # (B,N)


class SpatioTemporalTransformer(nn.Module):
    """
    Full model:
      - Encoder-only Transformer
      - Spatial PE = Laplacian eigenvectors
      - Temporal PE = relative + time-of-day + day-of-week
      - Dự đoán congestion tương lai cho anchor (node index anchor_idx)
    """
    def __init__(
        self,
        d_in,
        d_model=128,
        d_spa=16,
        nhead=4,
        num_layers=3,
        dim_feedforward=512,
        dropout=0.1,
        hidden_dim_head=128,
        anchor_idx=0,
    ):
        super().__init__()
        self.anchor_idx = anchor_idx

        self.spatial_enc = SpatialEncoding(d_model=d_model, d_spa=d_spa)

        self.encoder = SpatioTemporalEncoder(
            d_in=d_in,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
        )

        self.head = PredictionHead(d_model=d_model, hidden_dim=hidden_dim_head)

    def forward(self, x, time_of_day, day_of_week, lap_eigvec):
        # lap_eigvec: (B, N, d_spa)
        spatial_pe = self.spatial_enc(lap_eigvec)   # (B,N,d_model)
        H = self.encoder(x, time_of_day, day_of_week, spatial_pe)
        H_last = H[:, -1, :, :]     # (B,N,d_model)
        logits = self.head(H_last)  # (B,N)
        return logits


In [6]:
BASE_DIR = Path.cwd().parents[2]
OUTPUT_DIR = BASE_DIR / "data" / "processed" / "tomtom_stats"


SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)


In [7]:
# class ZoneDataset(Dataset):
#     def __init__(
#         self,
#         output_dir,
#         split_indices,
#         L=4,
#         delta=1,
#         tau_max=10,
#         Wmin=0.5,
#         top_k=60,
#         focus_prob=0.0,     # <--- thêm
#         focus_on="t+delta", # <--- thêm: "t" hoặc "t+delta"
#     ):
#         self.output_dir = Path(output_dir)
#         data = load_traffic_tensor(self.output_dir)
#
#         self.values = data["values"]        # (T,N)
#         self.segment_ids = data["segment_ids"]
#         self.T = self.values.shape[0]
#
#         meta = np.load(self.output_dir / "traffic_tensor.npz")
#         self.time_of_day = meta["time_of_day"]
#         self.day_of_week = meta["day_of_week"]
#         self.is_congested = meta["is_congested"]
#
#         self.L = L
#         self.delta = delta
#         self.split_indices = split_indices
#
#         self.tau_max = tau_max
#         self.Wmin = Wmin
#         self.top_k = top_k
#
#         self.focus_prob = float(focus_prob)
#         self.focus_on = focus_on
#     def __len__(self):
#         return len(self.split_indices)
#
#     def __getitem__(self, idx):
#         t = self.split_indices[idx]
#
#         if self.focus_prob > 0 and np.random.rand() < self.focus_prob:
#             tt = t + self.delta if self.focus_on == "t+delta" else t
#             cong = self.is_congested[tt, :]              # (N,)
#             cand = np.where(cong == 1)[0]                # indices trong [0..N-1]
#             if cand.size > 0:
#                 seed_idx = int(np.random.choice(cand))
#                 seed_seg_id = int(self.segment_ids[seed_idx])
#             else:
#                 seed_seg_id = int(np.random.choice(self.segment_ids))
#         else:
#             seed_seg_id = int(np.random.choice(self.segment_ids))
#
#         zone = build_zone(
#             output_dir=self.output_dir,
#             seed_seg_id=seed_seg_id,
#             tau_max=self.tau_max,
#             Wmin=self.Wmin,
#             top_k=self.top_k,
#             d_spa=16,
#         )
#
#         zone_idx = zone["zone_indices"]        # (N_zone,)
#         lap_eigvec = zone["lap_eigvec"]        # (N_zone, d_spa)
#
#         # ------------- Build time window --------------
#         x = self.values[t-self.L+1 : t+1, zone_idx]  # (L, N_zone)
#         tod = self.time_of_day[t-self.L+1 : t+1]     # (L,)
#         dow = self.day_of_week[t-self.L+1 : t+1]     # (L,)
#
#         # (optionally expand dims for d_in = 1)
#         x = x[..., None]  # (L,N_zone,1)
#
#         # label y = congestion at (t+delta)
#         y = self.is_congested[t + self.delta, zone_idx].astype(np.float32)
#         return {
#             "x": torch.tensor(x, dtype=torch.float32),
#             "tod": torch.tensor(tod, dtype=torch.float32),
#             "dow": torch.tensor(dow, dtype=torch.float32),
#             "lap": torch.tensor(lap_eigvec, dtype=torch.float32),
#             "y": torch.tensor(y, dtype=torch.float32),
#         }


In [8]:
class ZoneDataset(Dataset):
    def __init__(self, output_dir, split_indices, L=4, delta=1,
                 tau_max=10, Wmin=0.5, top_k=60,mode="train", p_random=0.5, p_cong=0.3):

        self.output_dir = Path(output_dir)
        data = load_traffic_tensor(self.output_dir)

        self.values = data["values"]
        self.segment_ids = data["segment_ids"]
        self.is_congested = data["is_congested"]
        self.time_of_day = data["time_of_day"]
        self.day_of_week = data["day_of_week"]

        self.split_indices = split_indices
        self.L = L
        self.delta = delta
        self.tau_max = tau_max
        self.Wmin = Wmin
        self.top_k = top_k

        # ---- for hybrid sampling ----
        self.cong_freq = self.is_congested.mean(axis=0)
        self.top_corr_idx = np.argsort(-self.cong_freq)[:200]

        self.p_random = p_random
        self.p_cong = p_cong
        self.mode = mode

    def __len__(self):
        return len(self.split_indices)

    def _seed_random(self, rng):
        return int(rng.choice(self.segment_ids))

    def _seed_congested(self, t, rng):
        idxs = np.where(self.is_congested[t] == 1)[0]
        if len(idxs) == 0:
            return self._seed_random(rng)
        return int(self.segment_ids[int(rng.choice(idxs))])

    def _seed_high_wpos(self, rng):
        idx = int(rng.choice(self.top_corr_idx))
        return int(self.segment_ids[idx])

    def _choose_seed_eval(self, t):
        rng = np.random.RandomState(int(t) + 12345)
        p = rng.rand()
        if p < self.p_random: return self._seed_random(rng)
        elif p < self.p_random + self.p_cong: return self._seed_congested(t, rng)
        else: return self._seed_high_wpos(rng)

    def _choose_seed_train(self, t):
        p = np.random.rand()
        if p < self.p_random:
            return int(np.random.choice(self.segment_ids))
        elif p < self.p_random + self.p_cong:
            idxs = np.where(self.is_congested[t] == 1)[0]
            if len(idxs) == 0:
                return int(np.random.choice(self.segment_ids))
            return int(self.segment_ids[int(np.random.choice(idxs))])
        else:
            idx = int(np.random.choice(self.top_corr_idx))
            return int(self.segment_ids[idx])

    def __getitem__(self, idx):
        t = self.split_indices[idx]

        if self.mode == "train":
            seed_seg_id = self._choose_seed_train(t)
        else:
            seed_seg_id = self._choose_seed_eval(t)


        zone = build_zone(
            output_dir=self.output_dir,
            seed_seg_id=seed_seg_id,
            tau_max=self.tau_max,
            Wmin=self.Wmin,
            top_k=self.top_k,
            d_spa=16,
        )

        zone_idx = zone["zone_indices"]
        lap_eigvec = zone["lap_eigvec"]

        x = self.values[t-self.L+1:t+1, zone_idx][..., None]
        y = self.is_congested[t+self.delta, zone_idx]

        return {
            "x": torch.tensor(x, dtype=torch.float32),
            "tod": torch.tensor(self.time_of_day[t-self.L+1:t+1]),
            "dow": torch.tensor(self.day_of_week[t-self.L+1:t+1]),
            "lap": torch.tensor(lap_eigvec, dtype=torch.float32),
            "y": torch.tensor(y, dtype=torch.float32),
        }


In [9]:
# Cell: Collate function để pad theo N_zone lớn nhất trong batch
def collate_zone(batch):
    """
    batch = list of dict { x:(L,N_zone,1), tod:(L,), dow:(L,), lap:(N_zone,d_spa), y:(N_zone,) }
    Trả về tensors padded theo N_zone_max
    """
    L = batch[0]["x"].shape[0]
    d_in = batch[0]["x"].shape[-1]
    d_spa = batch[0]["lap"].shape[-1]

    # ---- tìm max N_zone trong batch
    Nmax = max(item["x"].shape[1] for item in batch)
    B = len(batch)

    # ---- init tensor padded
    x_pad  = torch.zeros(B, L, Nmax, d_in)
    lap_pad= torch.zeros(B, Nmax, d_spa)
    y_pad  = torch.full((B, Nmax), fill_value=-1.0)  # dùng -1 để mask label
    mask   = torch.zeros(B, Nmax)                    # 1=valid, 0=padded

    tod = torch.stack([item["tod"] for item in batch], dim=0)  # (B,L)
    dow = torch.stack([item["dow"] for item in batch], dim=0)  # (B,L)

    # ---- copy từng sample vào padded tensor
    for i, item in enumerate(batch):
        Ni = item["x"].shape[1]
        x_pad[i, :, :Ni, :] = item["x"]
        lap_pad[i, :Ni, :] = item["lap"]
        y_pad[i, :Ni] = item["y"]
        mask[i, :Ni] = 1.0

    return {
        "x": x_pad,
        "tod": tod,
        "dow": dow,
        "lap": lap_pad,
        "y": y_pad,
        "mask": mask
    }


In [10]:
def masked_bce_loss(logits, labels, mask):
    """
    logits: (B,N)
    labels: (B,N)
    mask:   (B,N)  — 1=valid, 0=padding
    """
    loss = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    loss = loss * mask
    return loss.sum() / (mask.sum() + 1e-6)

@torch.no_grad()
def masked_metrics_from_logits(logits, labels, mask, threshold=0.5):
    """
    logits: (B,N)
    labels: (B,N) 0/1
    mask:   (B,N) 1=valid, 0=pad
    """
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    # chỉ lấy phần valid
    m = mask > 0.5
    y = labels[m]
    p = preds[m]

    # confusion
    tp = ((p == 1) & (y == 1)).sum().float()
    tn = ((p == 0) & (y == 0)).sum().float()
    fp = ((p == 1) & (y == 0)).sum().float()
    fn = ((p == 0) & (y == 1)).sum().float()

    acc = (tp + tn) / (tp + tn + fp + fn + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    rec = tp / (tp + fn + 1e-9)

    return acc.item(), prec.item(), rec.item()

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    t0 = time.perf_counter()

    for it, batch in enumerate(loader, start=1):
        if it % 2 == 0:
            print(f"[train] iter {it}/{len(loader)} elapsed={time.perf_counter()-t0:.1f}s", flush=True)

        x   = batch["x"].to(device, non_blocking=True)
        tod = batch["tod"].to(device, non_blocking=True)
        dow = batch["dow"].to(device, non_blocking=True)
        lap = batch["lap"].to(device, non_blocking=True)
        y   = batch["y"].to(device, non_blocking=True)
        m   = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(x, tod, dow, lap, node_mask=m)  # (B,N)

        loss = masked_bce_loss(logits, y, m)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

@torch.no_grad()
def eval_epoch(model, loader, device, threshold=0.5):
    model.eval()
    total_loss = 0.0

    TP = TN = FP = FN = 0.0

    for batch in loader:
        x   = batch["x"].to(device)
        tod = batch["tod"].to(device)
        dow = batch["dow"].to(device)
        lap = batch["lap"].to(device)
        y   = batch["y"].to(device)
        m   = batch["mask"].to(device)

        logits = model(x, tod, dow, lap, node_mask=m)
        loss = masked_bce_loss(logits, y, m)
        total_loss += loss.item()

        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()
        mm = m > 0.5
        yy = y[mm]
        pp = preds[mm]

        TP += ((pp == 1) & (yy == 1)).sum().item()
        TN += ((pp == 0) & (yy == 0)).sum().item()
        FP += ((pp == 1) & (yy == 0)).sum().item()
        FN += ((pp == 0) & (yy == 1)).sum().item()


    acc  = (TP + TN) / (TP + TN + FP + FN + 1e-9)
    prec = TP / (TP + FP + 1e-9)
    rec  = TP / (TP + FN + 1e-9)
    f1   = (2 * prec * rec) / (prec + rec + 1e-9)
    print("TP,FP,FN =", TP, FP, FN)
    print("prec raw =", prec)                # Python float
    print("prec sci =", f"{prec:.8e}")       # dạng scientific
    return total_loss / len(loader), acc, prec, rec, f1




In [11]:
import time
import os
def build_splits(output_dir, L=3, delta=1):
    data = np.load(Path(output_dir) / "traffic_tensor.npz")
    T = data["values"].shape[0]

    # -------- ratio 80 / 10 / 10 ----------
    train_end = int(T * 0.80)
    valid_end = int(T * 0.90)

    def valid_indices(start, end):
        idxs = []
        for t in range(start, end):
            if (t - L + 1) < 0:
                continue
            if (t + delta) >= end:
                continue
            idxs.append(t)
        return idxs

    split_train = valid_indices(0, train_end)
    split_valid = valid_indices(train_end, valid_end)
    split_test  = valid_indices(valid_end, T)

    print("Total T =", T)
    print(f"Train={len(split_train)}, Valid={len(split_valid)}, Test={len(split_test)}")

    return split_train, split_valid, split_test
split_train, split_valid, split_test = build_splits(OUTPUT_DIR, L=3, delta=1)

train_ds = ZoneDataset(OUTPUT_DIR, split_train, L=3, delta=1, mode="train", p_random=0.5, p_cong=0.3)
valid_ds = ZoneDataset(OUTPUT_DIR, split_valid, L=3, delta=1, mode="eval",  p_random=0.5, p_cong=0.3)
test_ds  = ZoneDataset(OUTPUT_DIR, split_test,  L=3, delta=1, mode="eval",  p_random=0.5, p_cong=0.3)

# thêm train_eval để eval ổn định (khuyên dùng)
train_eval_ds = ZoneDataset(OUTPUT_DIR, split_train, L=3, delta=1, mode="eval", p_random=0.5, p_cong=0.3)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  collate_fn=collate_zone, pin_memory=True)
train_eval_loader = DataLoader(train_eval_ds, batch_size=8, shuffle=False, collate_fn=collate_zone,pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False, collate_fn=collate_zone,pin_memory=True)
# num_workers = min(8, os.cpu_count() or 1)

test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, collate_fn=collate_zone)

model = SpatioTemporalTransformer(
    d_in=1, d_model=128, d_spa=16, nhead=4, num_layers=3
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
EPOCHS = 20
CKPT_DIR = BASE_DIR / "saved_models" / "checkpoints3"
CKPT_DIR.mkdir(parents=True, exist_ok=True)


Total T = 528
Train=419, Valid=52, Test=52


In [12]:
import time, math
from pathlib import Path
import torch

def fit(
    model,
    train_loader,
    train_eval_loader,
    valid_loader,
    test_loader,
    optimizer,
    device,
    epochs=20,
    ckpt_dir=Path("checkpoints"),
    threshold=0.5,
    select_by="valid_loss",  # "valid_loss" hoặc "valid_f1"
):
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    best_score = None
    best_path = None

    history = []

    for epoch in range(1, epochs + 1):
        print(f"\n=== START EPOCH {epoch}/{epochs} ===", flush=True)
        print("[DBG] about to fetch FIRST batch from train_loader", flush=True)
        t_fetch0 = time.time()
        it = iter(train_loader)
        batch0 = next(it)
        print(f"[DBG] got FIRST batch in {time.time()-t_fetch0:.1f}s", flush=True)
        t0 = time.time()

        # -------- train --------
        model.train()
        total_loss = 0.0

        for it, batch in enumerate(train_loader, start=1):
            if it == 1:
                print("[FIT] got FIRST batch from train_loader", flush=True)
            if it % 2 == 0:
                print(
                    f"[TRAIN] epoch={epoch:02d} "
                    f"iter={it}/{len(train_loader)}",
                    flush=True
                )
            x   = batch["x"].to(device, non_blocking=True)
            tod = batch["tod"].to(device, non_blocking=True)
            dow = batch["dow"].to(device, non_blocking=True)
            lap = batch["lap"].to(device, non_blocking=True)
            y   = batch["y"].to(device, non_blocking=True)
            m   = batch["mask"].to(device, non_blocking=True)
            if it == 1:
                print("[TRAIN] data moved to device", flush=True)
            optimizer.zero_grad()
            logits = model(x, tod, dow, lap)
            loss = masked_bce_loss(logits, y, m)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)

        # -------- eval train & valid --------
        tr_loss, tr_acc, tr_prec, tr_rec, tr_f1 = eval_epoch(
    model, train_eval_loader, device, threshold=threshold
)

        va_loss, va_acc, va_prec, va_rec, va_f1 = eval_epoch(
            model, valid_loader, device, threshold=threshold
        )

        dt = time.time() - t0

        print(
            f"Epoch {epoch:02d}/{epochs} | time={dt/60:.1f}m | "
            f"TrainLoss={train_loss:.4f} | "
            f"Train Eval: loss={tr_loss:.4f} acc={tr_acc:.4f} p={tr_prec:.4f} r={tr_rec:.4f} f1={tr_f1:.4f} | "
            f"Valid Eval: loss={va_loss:.4f} acc={va_acc:.4f} p={va_prec:.4f} r={va_rec:.4f} f1={va_f1:.4f}"
        )

        # -------- save ckpt mỗi epoch (optional) --------
        ckpt_path = ckpt_dir / f"epoch_{epoch:03d}.pt"
        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "train_loss": train_loss,
                "train_eval": (tr_loss, tr_acc, tr_prec, tr_rec, tr_f1),
                "valid_eval": (va_loss, va_acc, va_prec, va_rec, va_f1),
                "threshold": threshold,
            },
            ckpt_path,
        )

        # -------- chọn best --------
        if select_by == "valid_loss":
            score = -va_loss              # loss càng thấp càng tốt => lấy âm để maximize
        elif select_by == "valid_f1":
            score = va_f1                 # f1 càng cao càng tốt
        else:
            raise ValueError("select_by must be 'valid_loss' or 'valid_f1'")

        if (best_score is None) or (score > best_score):
            best_score = score
            best_path = ckpt_dir / "best.pt"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "train_loss": train_loss,
                    "train_eval": (tr_loss, tr_acc, tr_prec, tr_rec, tr_f1),
                    "valid_eval": (va_loss, va_acc, va_prec, va_rec, va_f1),
                    "threshold": threshold,
                    "select_by": select_by,
                },
                best_path,
            )
            print(f"  ✅ BEST updated @ epoch {epoch:02d} -> {best_path}")

        history.append(
            dict(
                epoch=epoch,
                train_loss=train_loss,
                tr_loss=tr_loss, tr_acc=tr_acc, tr_prec=tr_prec, tr_rec=tr_rec, tr_f1=tr_f1,
                va_loss=va_loss, va_acc=va_acc, va_prec=va_prec, va_rec=va_rec, va_f1=va_f1,
            )
        )

    # -------- load best & final eval on train/valid/test --------
    print("\n===== Load BEST checkpoint and evaluate on TRAIN/VALID/TEST =====")
    best = torch.load(best_path, map_location="cpu")
    model.load_state_dict(best["model_state"])
    model = model.to(device)
    model.eval()

    tr = eval_epoch(model, train_loader, device, threshold=threshold)
    va = eval_epoch(model, valid_loader, device, threshold=threshold)
    te = eval_epoch(model, test_loader,  device, threshold=threshold)

    print(f"BEST ckpt: {best_path} (epoch {best['epoch']}) | select_by={select_by}")
    print(f"TRAIN: loss={tr[0]:.4f} acc={tr[1]:.4f} p={tr[2]:.4f} r={tr[3]:.4f} f1={tr[4]:.4f}")
    print(f"VALID: loss={va[0]:.4f} acc={va[1]:.4f} p={va[2]:.4f} r={va[3]:.4f} f1={va[4]:.4f}")
    print(f"TEST : loss={te[0]:.4f} acc={te[1]:.4f} p={te[2]:.4f} r={te[3]:.4f} f1={te[4]:.4f}")

    return best_path, history


In [13]:
best_path, history = fit(
    model=model,
    train_loader=train_loader,
    train_eval_loader = train_eval_loader,
    valid_loader=valid_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    device=device,
    epochs=20,
    ckpt_dir=CKPT_DIR,
    threshold=0.5,
    select_by="valid_f1",   # hoặc "valid_loss"
)



=== START EPOCH 1/20 ===
[DBG] about to fetch FIRST batch from train_loader
[DBG] got FIRST batch in 24.5s
[FIT] got FIRST batch from train_loader
[TRAIN] data moved to device


C:\Users\ADMIN\miniconda3\envs\Specialized_Project_torchdml\lib\site-packages\torch\nn\functional.py:3244: UserWarning: The operator 'aten::log_sigmoid_forward' is not currently supported on the DML backend and will fall back to run on the CPU. This may have performance implications. (Triggered internally at C:\__w\1\s\pytorch-directml-plugin\torch_directml\csrc\dml\dml_cpu_fallback.cpp:17.)
  return torch.binary_cross_entropy_with_logits(input, target, weight, pos_weight, reduction_enum)


[TRAIN] epoch=01 iter=2/53
[TRAIN] epoch=01 iter=4/53
[TRAIN] epoch=01 iter=6/53
[TRAIN] epoch=01 iter=8/53
[TRAIN] epoch=01 iter=10/53
[TRAIN] epoch=01 iter=12/53
[TRAIN] epoch=01 iter=14/53
[TRAIN] epoch=01 iter=16/53
[TRAIN] epoch=01 iter=18/53
[TRAIN] epoch=01 iter=20/53
[TRAIN] epoch=01 iter=22/53
[TRAIN] epoch=01 iter=24/53
[TRAIN] epoch=01 iter=26/53
[TRAIN] epoch=01 iter=28/53
[TRAIN] epoch=01 iter=30/53
[TRAIN] epoch=01 iter=32/53
[TRAIN] epoch=01 iter=34/53
[TRAIN] epoch=01 iter=36/53
[TRAIN] epoch=01 iter=38/53
[TRAIN] epoch=01 iter=40/53
[TRAIN] epoch=01 iter=42/53
[TRAIN] epoch=01 iter=44/53
[TRAIN] epoch=01 iter=46/53
[TRAIN] epoch=01 iter=48/53
[TRAIN] epoch=01 iter=50/53
[TRAIN] epoch=01 iter=52/53
Epoch 01/20 | time=54.4m | TrainLoss=0.5903 | Train Eval: loss=0.5580 acc=0.7236 p=0.1429 r=0.0002 f1=0.0004 | Valid Eval: loss=0.5857 acc=0.6806 p=0.0000 r=0.0000 f1=0.0000
  ✅ BEST updated @ epoch 01 -> C:\AI\Specialized_Project2_github\Urban-Traffic-Links\saved_models\chec

KeyboardInterrupt: 